# Extraction batch de PDFs → Parquet
Les PDFs sont traités en parallèle via `PdfTwoColumnExtractor` (analyse.py).  
Le résultat est un fichier Parquet avec 3 colonnes : `pdf_file`, `titre`, `description`.  
Les PDFs en erreur sont inclus dans le Parquet avec `titre = "ERROR"`.

In [ ]:
# ============================================================
#  IMPORTS  — si cette cellule passe, tout le reste déroule
# ============================================================
import sys
import os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# analyse.py doit être dans le même dossier que ce notebook
# (ou adapter le chemin ci-dessous)
PROJET_DIR = Path("/Users/orsopaoli/Desktop/BNP/BNP_app/Conversion info TS de PDF a python")
if str(PROJET_DIR) not in sys.path:
    sys.path.insert(0, str(PROJET_DIR))

from analyse import PdfTwoColumnExtractor   # vérification à l'import

print("Imports OK")

In [ ]:
# ============================================================
#  CHEMINS — modifier ici
# ============================================================

# Dossier contenant tous les PDFs (cherche récursivement)
PDF_DIR = Path("/chemin/vers/vos/pdfs")   # ← À MODIFIER

# Fichier Parquet de sortie
OUTPUT_PARQUET = Path("/chemin/vers/sortie/extractions.parquet")   # ← À MODIFIER
OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

# Dossier temporaire pour les batches (supprimé après consolidation)
BATCH_DIR = OUTPUT_PARQUET.parent / "_batches"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 500                                        # PDFs par fichier batch
MAX_WORKERS = max(1, (os.cpu_count() or 4) // 2)        # workers parallèles

# Lister les PDFs
all_pdfs = sorted(PDF_DIR.rglob("*.pdf"))
print(f"{len(all_pdfs)} PDFs trouvés")
print(f"MAX_WORKERS = {MAX_WORKERS}  |  BATCH_SIZE = {BATCH_SIZE}")

In [ ]:
# ============================================================
#  WORKER  (les imports sont répétés à l'intérieur : obligatoire
#  pour que ProcessPoolExecutor puisse sérialiser la fonction)
# ============================================================

def extract_one_pdf(args):
    """
    Extrait un PDF et retourne un DataFrame avec colonnes
    [pdf_file, titre, description].
    En cas d'erreur, retourne une ligne avec titre='ERROR'.
    """
    pdf_path_str, projet_dir_str = args
    import sys
    from pathlib import Path
    import pandas as pd
    if projet_dir_str not in sys.path:
        sys.path.insert(0, projet_dir_str)
    from analyse import PdfTwoColumnExtractor

    filename = Path(pdf_path_str).name
    try:
        extractor = PdfTwoColumnExtractor()
        df = extractor.to_dataframe(pdf_path_str)
        if df.empty:
            raise ValueError("DataFrame vide après extraction")
        df.insert(0, "pdf_file", filename)
        return df
    except Exception as exc:
        return pd.DataFrame([{
            "pdf_file":    filename,
            "titre":       "ERROR",
            "description": f"{type(exc).__name__}: {exc}",
        }])

print("Worker défini")

In [ ]:
# ============================================================
#  EXTRACTION BATCH
# ============================================================

SCHEMA = pa.schema([
    pa.field("pdf_file",    pa.string()),
    pa.field("titre",       pa.string()),
    pa.field("description", pa.string()),
])

def batched(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]

args_list   = [(str(p), str(PROJET_DIR)) for p in all_pdfs]
total_rows  = 0
total_errors = 0

with tqdm(total=len(all_pdfs), desc="PDFs", unit="pdf") as pbar:
    for batch_idx, batch in enumerate(batched(args_list, BATCH_SIZE)):
        results = []

        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(extract_one_pdf, a): a for a in batch}
            for future in as_completed(futures):
                df = future.result()
                results.append(df)
                n_err = (df["titre"] == "ERROR").sum()
                total_errors += n_err
                total_rows   += len(df) - n_err
                pbar.update(1)
                pbar.set_postfix(lignes=total_rows, erreurs=total_errors)

        batch_df = pd.concat(results, ignore_index=True)
        for col in ["titre", "description"]:
            batch_df[col] = batch_df[col].fillna("").astype(str)

        table = pa.Table.from_pandas(batch_df, schema=SCHEMA, preserve_index=False)
        pq.write_table(table, BATCH_DIR / f"batch_{batch_idx:06d}.parquet", compression="snappy")

print(f"\nTerminé — {total_rows:,} lignes extraites, {total_errors} erreurs")

In [ ]:
# ============================================================
#  CONSOLIDATION → un seul fichier Parquet final
# ============================================================

df_all = pd.read_parquet(BATCH_DIR)
df_all.to_parquet(OUTPUT_PARQUET, index=False, compression="snappy")

size_mb = OUTPUT_PARQUET.stat().st_size / 1e6
print(f"{len(df_all):,} lignes  |  {df_all['pdf_file'].nunique():,} PDFs  |  {size_mb:.1f} Mo")
print(f"Fichier : {OUTPUT_PARQUET}")

# Supprimer les fichiers batch intermédiaires
import shutil
shutil.rmtree(BATCH_DIR)
print("Batches intermédiaires supprimés")

In [ ]:
# ============================================================
#  VÉRIFICATION
# ============================================================

df = pd.read_parquet(OUTPUT_PARQUET)

errors = df[df["titre"] == "ERROR"]
print(f"Lignes totales  : {len(df):,}")
print(f"PDFs en erreur  : {errors['pdf_file'].nunique()}")

if len(errors):
    print("\nTop erreurs :")
    print(errors["description"].value_counts().head(10))

print("\nEchantillon données extraites :")
df[df["titre"] != "ERROR"].sample(min(10, len(df)))